# Back Translation Augmentation (VI -> EN -> VI)

## 1. Dependencies & Kết nối Google Drive

In [1]:
!pip install -q transformers sacremoses py_vncorenlp pandas torch

import os
import re
import unicodedata
from pathlib import Path
import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer
import py_vncorenlp
from google.colab import drive

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.2 MB/s eta 0:00:00
Device: cuda


In [2]:
drive.mount('/content/drive')

PROJECT_DIR = Path("/content/drive/MyDrive/Hate_Speech_Detection")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
AUGMENTED_DIR = PROJECT_DIR / "data" / "augmented"
AUGMENTED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_DIR}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/Hate_Speech_Detection


## 2. VnCoreNLP -- Phân từ lại sau khi dịch

Kết quả dịch ngược trả về câu tiếng Việt thô (chưa phân từ) -- cần phân từ lại để khớp schema với `train.csv` (cột `text` đã phân từ).

In [3]:
model_dir = '/content/vncorenlp'
if 'rdrsegmenter' not in globals():
    if not os.path.exists(model_dir):
        os.makedirs(model_dir, exist_ok=True)
        py_vncorenlp.download_model(save_dir=model_dir)
    rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=model_dir)

def segment_text(text):
    try:
        sentences = rdrsegmenter.word_segment(text)
        return " ".join(sentences)
    except Exception:
        return text

def text_key(text):
    text = unicodedata.normalize("NFKC", str(text)).strip().lower()
    return re.sub(r"\s+", " ", text)

## 3. Load Model Dịch (Helsinki-NLP MarianMT)

2 model riêng cho 2 chiều dịch: `vi-en` và `en-vi`.

In [4]:
model_vi_en_name = "Helsinki-NLP/opus-mt-vi-en"
model_en_vi_name = "Helsinki-NLP/opus-mt-en-vi"

tokenizer_vi_en = MarianTokenizer.from_pretrained(model_vi_en_name)
model_vi_en = MarianMTModel.from_pretrained(model_vi_en_name).to(device)
tokenizer_en_vi = MarianTokenizer.from_pretrained(model_en_vi_name)
model_en_vi = MarianMTModel.from_pretrained(model_en_vi_name).to(device)

@torch.no_grad()
def back_translate(texts):
    inputs = tokenizer_vi_en(texts, return_tensors="pt", padding=True, truncation=True).to(device)
    translated_en = model_vi_en.generate(**inputs)
    en_texts = tokenizer_vi_en.batch_decode(translated_en, skip_special_tokens=True)

    inputs_en = tokenizer_en_vi(en_texts, return_tensors="pt", padding=True, truncation=True).to(device)
    translated_vi = model_en_vi.generate(**inputs_en)
    vi_texts = tokenizer_en_vi.batch_decode(translated_vi, skip_special_tokens=True)

    return vi_texts

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  289MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

## 4. Load Dữ liệu -- chỉ lấy 2 lớp thiểu số (HATE, OFFENSIVE)

In [5]:
train = pd.read_csv(PROCESSED_DIR / "train.csv")
dev = pd.read_csv(PROCESSED_DIR / "dev.csv")
test = pd.read_csv(PROCESSED_DIR / "test.csv")

if 'text_raw' not in train.columns:
    train['text_raw'] = train['text'].astype(str).str.replace('_', ' ')
    train['text_raw'] = train['text_raw'].str.replace(r'\s+', ' ', regex=True).str.strip()

df_train = train.dropna(subset=['text', 'text_raw']).reset_index(drop=True)
df_train['text_raw'] = df_train['text_raw'].astype(str)

df_minority = df_train[df_train['label'].isin(['HATE', 'OFFENSIVE'])].copy()

# Chạy theo batch
batch_size = 16
bt_results = []
texts_to_translate = df_minority['text_raw'].tolist()

print(f"Bắt đầu dịch {len(texts_to_translate)} câu...")

df_minority = train[train["label"].isin(["HATE", "OFFENSIVE"])].reset_index(drop=True)
print(f"Số câu cần back-translate: {len(df_minority):,}")
print(df_minority["label"].value_counts())

Bắt đầu dịch 25308 câu...
Số câu cần back-translate: 25,309
label
OFFENSIVE    19577
HATE          5732
Name: count, dtype: int64


## 5. Chạy Back-translation

In [6]:
BATCH_SIZE = 16
CHECKPOINT_PATH = AUGMENTED_DIR / "_bt_checkpoint.csv"

bt_results = []
start_idx = 0
texts_to_translate = df_minority["text_raw"].tolist()

if CHECKPOINT_PATH.exists():
    saved = pd.read_csv(CHECKPOINT_PATH)
    bt_results = saved["bt_text"].tolist()
    start_idx = len(bt_results)
    print(f"Tìm thấy checkpoint -- đã dịch {start_idx:,}/{len(texts_to_translate):,}, tiếp tục từ đó.")

print(f"Bắt đầu dịch {len(texts_to_translate):,} câu...")
for i in range(start_idx, len(texts_to_translate), BATCH_SIZE):
    batch = texts_to_translate[i:i + BATCH_SIZE]
    try:
        bt_results.extend(back_translate(batch))
    except Exception as e:
        print(f"  Lỗi ở batch {i}: {e} -- giữ nguyên câu gốc cho batch này")
        bt_results.extend(batch)

    if (i // BATCH_SIZE) % 10 == 0:
        pd.DataFrame({"bt_text": bt_results}).to_csv(CHECKPOINT_PATH, index=False)
        print(f"  Đã dịch: {len(bt_results):,}/{len(texts_to_translate):,}")

pd.DataFrame({"bt_text": bt_results}).to_csv(CHECKPOINT_PATH, index=False)
print(f"Hoàn tất dịch: {len(bt_results):,} câu")

Tìm thấy checkpoint -- đã dịch 21,936/25,309, tiếp tục từ đó.
Bắt đầu dịch 25,309 câu...
  Đã dịch: 22,096/25,309
  Đã dịch: 22,256/25,309
  Đã dịch: 22,416/25,309
  Đã dịch: 22,576/25,309
  Đã dịch: 22,736/25,309
  Đã dịch: 22,896/25,309
  Đã dịch: 23,056/25,309
  Đã dịch: 23,216/25,309
  Đã dịch: 23,376/25,309
  Đã dịch: 23,536/25,309
  Đã dịch: 23,696/25,309
  Đã dịch: 23,856/25,309
  Đã dịch: 24,016/25,309
  Đã dịch: 24,176/25,309
  Đã dịch: 24,336/25,309
  Đã dịch: 24,496/25,309
  Đã dịch: 24,656/25,309
  Đã dịch: 24,816/25,309
  Đã dịch: 24,976/25,309
  Đã dịch: 25,136/25,309
  Đã dịch: 25,296/25,309
Hoàn tất dịch: 25,309 câu


## 6. Lọc câu Dịch "No-op" (không thay đổi so với gốc)

Nếu model dịch thất bại hoặc câu quá ngắn/toàn slang không dịch được, kết quả có thể giống hệt câu gốc

In [7]:
df_bt = pd.DataFrame({
    "text_raw": bt_results,
    "label": df_minority["label"].tolist(),
    "source": "bt",
    "_orig_text_raw": df_minority["text_raw"].tolist(),
})

df_bt["_key"] = df_bt["text_raw"].map(text_key)
df_bt["_orig_key"] = df_bt["_orig_text_raw"].map(text_key)

noop_mask = df_bt["_key"] == df_bt["_orig_key"]
print(f"Câu dịch không đổi so với gốc (loại bỏ): {noop_mask.sum():,} / {len(df_bt):,}")

df_bt = df_bt[~noop_mask].reset_index(drop=True)
df_bt = df_bt.drop_duplicates(subset=["_key"]).reset_index(drop=True)
print(f"Còn lại sau khi loại no-op + trùng lặp nội bộ: {len(df_bt):,}")

Câu dịch không đổi so với gốc (loại bỏ): 0 / 25,309
Còn lại sau khi loại no-op + trùng lặp nội bộ: 24,286


In [8]:
def has_repetition_loop(text, min_repeat=4, phrase_words=2):
    """Phát hiện cụm từ lặp lại liên tiếp nhiều lần -- dấu hiệu MT bị lỗi decode."""
    words = str(text).split()
    if len(words) < min_repeat * phrase_words:
        return False
    for i in range(len(words) - min_repeat * phrase_words + 1):
        phrase = words[i:i + phrase_words]
        repeat_count = 1
        j = i + phrase_words
        while j + phrase_words <= len(words) and words[j:j + phrase_words] == phrase:
            repeat_count += 1
            j += phrase_words
        if repeat_count >= min_repeat:
            return True
    return False

repetition_mask = df_bt["text_raw"].apply(has_repetition_loop)
print(f"Bị lỗi lặp vô hạn (loại bỏ): {repetition_mask.sum():,} / {len(df_bt):,}")

df_bt = df_bt[~repetition_mask].reset_index(drop=True)
print(f"Còn lại: {len(df_bt):,}")

Bị lỗi lặp vô hạn (loại bỏ): 585 / 24,286
Còn lại: 23,701


In [10]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("bkai-foundation-models/vietnamese-bi-encoder")

df_bt = df_bt.dropna(subset=["_orig_text_raw", "text_raw"]).reset_index(drop=True)

orig_segmented = df_bt["_orig_text_raw"].astype(str).apply(segment_text).astype(str).tolist()
bt_segmented = df_bt["text_raw"].astype(str).apply(segment_text).astype(str).tolist()

print(f"Bắt đầu encode {len(orig_segmented)} cặp câu để so sánh độ tương đồng...")

orig_emb = embedder.encode(orig_segmented, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
bt_emb = embedder.encode(bt_segmented, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

df_bt["semantic_similarity"] = (orig_emb * bt_emb).sum(axis=1)

SEMANTIC_THRESHOLD = 0.60
low_semantic_mask = df_bt["semantic_similarity"] < SEMANTIC_THRESHOLD

print(f"Similarity thấp (nghi mất nghĩa, loại bỏ): {low_semantic_mask.sum():,} / {len(df_bt):,}")
print("-" * 40)
print("Thống kê độ tương đồng:")
print(df_bt["semantic_similarity"].describe())
print("-" * 40)

df_bt = df_bt[~low_semantic_mask].reset_index(drop=True)
print(f"Còn lại đưa vào tập train: {len(df_bt):,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Bắt đầu encode 23700 cặp câu để so sánh độ tương đồng...


Batches:   0%|          | 0/371 [00:00<?, ?it/s]

Batches:   0%|          | 0/371 [00:00<?, ?it/s]

Similarity thấp (nghi mất nghĩa, loại bỏ): 19,466 / 23,700
----------------------------------------
Thống kê độ tương đồng:
count    23700.000000
mean         0.422378
std          0.187806
min         -0.220943
25%          0.292917
50%          0.427589
75%          0.554738
max          1.000000
Name: semantic_similarity, dtype: float64
----------------------------------------
Còn lại đưa vào tập train: 4,234


## 7. Leakage Check với Train/Dev/Test

Loại câu augmented trùng với dữ liệu đã có sẵn

In [12]:
for df in [train, dev, test]:
    if 'text_raw' not in df.columns:
        df['text_raw'] = df['text'].astype(str).str.replace('_', ' ')
        df['text_raw'] = df['text_raw'].str.replace(r'\s+', ' ', regex=True).str.strip()

    df['text_raw'] = df['text_raw'].astype(str)

train["_key"] = train["text_raw"].map(text_key)
dev["_key"] = dev["text_raw"].map(text_key)
test["_key"] = test["text_raw"].map(text_key)

existing_keys = set(train["_key"]) | set(dev["_key"]) | set(test["_key"])
leak_mask = df_bt["_key"].isin(existing_keys)
print(f"Trùng với train/dev/test (loại bỏ): {leak_mask.sum():,}")

df_bt = df_bt[~leak_mask].reset_index(drop=True)
print(f"Còn lại đưa vào tập train: {len(df_bt):,}")

Trùng với train/dev/test (loại bỏ): 0
Còn lại đưa vào tập train: 4,234


## 8. Phân từ lại

Đọc ngẫu nhiên 20-30 cặp (gốc, dịch ngược) để xác nhận: câu vẫn giữ đúng tính chất OFFENSIVE/HATE sau khi dịch

In [13]:
df_bt["text"] = df_bt["text_raw"].apply(segment_text)

sample_for_review = df_bt.sample(min(25, len(df_bt)), random_state=42)
for _, row in sample_for_review.iterrows():
    orig = df_minority.loc[df_minority["text_raw"] == row["_orig_text_raw"], "text_raw"]
    print(f"[{row['label']}]")
    print(f"  Gốc:        {row['_orig_text_raw']}")
    print(f"  Back-trans: {row['text_raw']}")
    print()

[OFFENSIVE]
  Gốc:        thật ra không phải giải phóng mà là do những người còn sót lại ở phía bắc muốn vào sg nhưng kẹt tường lửa của lính chế độ cũ
  Back-trans: Nó thực sự không phải là về giải phóng, mà là về phần còn lại của những người phương Bắc muốn đi vào sg, nhưng họ bị mắc kẹt trong tường thành của một đội quân cũ.

[OFFENSIVE]
  Gốc:        Thím chạy qua cái vnvc khác chứ Mà vậy nên e mới k có mua cả gói đấy, lởm khởm vcc via theNEXTvoz for iPhone
  Back-trans: Bạn đã chạy qua một cái khác, và đó là lý do tại sao tôi đã không mua toàn bộ gói, điều tệ hại bởi vì củaEXTvoz cho iPhone.

[OFFENSIVE]
  Gốc:        Minh anh quá giỏi con người đối diện quá tệ
  Back-trans: Ming, anh quá giỏi mặt người khác.

[OFFENSIVE]
  Gốc:        người không tiến hóa từ vượn mà, học thuyết đó xàm chết mẹ luôn ấy. kaka
  Back-trans: Người đàn ông không tiến hóa từ loài vượn cáo, giả thuyết đó hỏng rồi.

[OFFENSIVE]
  Gốc:        4giờ đã 27k view ròi, quá nhanh quá nguy hiểm 😆
  Back-trans: Bây

## 9. Lưu kết quả

In [14]:
output_cols = ["text", "text_raw", "label", "source"]
df_bt[output_cols].to_csv(AUGMENTED_DIR / "aug_bt.csv", index=False, encoding="utf-8-sig")
print(f"Đã lưu {len(df_bt):,} câu vào {AUGMENTED_DIR / 'aug_bt.csv'}")

if CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()

print("\nSo sánh trước/sau augmentation:")
before = df_minority["label"].value_counts()
after_added = df_bt["label"].value_counts()
comparison = pd.DataFrame({"train_gốc": before, "thêm_từ_bt": after_added}).fillna(0).astype(int)
comparison["tổng_sau_augment"] = comparison["train_gốc"] + comparison["thêm_từ_bt"]
comparison

Đã lưu 4,234 câu vào /content/drive/MyDrive/Hate_Speech_Detection/data/augmented/aug_bt.csv

So sánh trước/sau augmentation:


,train_gốc,thêm_từ_bt,tổng_sau_augment
label,,,
OFFENSIVE,19577,3407,22984
HATE,5732,827,6559
